# Post 008 — Naive Bayes & LDA: Generative Classifiers
## Dataset B: Silicon Defect Type Classification (Post-Silicon Validation)

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

When a die fails electrical test, the first question is: **what kind of failure is it?** Gate oxide breakdown? Metal short? Contact resistance? Each failure type has a distinct electrical signature — a pattern in leakage current, threshold voltage, and contact resistance measurements.

This notebook applies Gaussian Naive Bayes and LDA to classify silicon defect types from electrical test measurements, demonstrating how generative models can provide both fast classification and interpretable insights into what makes each defect type electrically unique.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

In [ ]:
df = pd.read_csv('../data/silicon_defect_classification.csv')
print(f'Shape: {df.shape}')
print(f'\nDefect type distribution:')
print(df['defect_type'].value_counts())
df.head()

## 1. Electrical Signature Analysis

Before modeling, let's understand what makes each defect type electrically distinct. This is the domain knowledge that Naive Bayes will encode as class-conditional distributions.

In [ ]:
feature_cols = [c for c in df.columns if c != 'defect_type']
defect_types = df['defect_type'].unique()
colors = plt.cm.Set1(np.linspace(0, 1, len(defect_types)))

# Violin plots: feature distributions per defect type
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    df.boxplot(column=feat, by='defect_type', ax=axes[i], 
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(feat, fontsize=9)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30, labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Electrical Feature Distributions by Defect Type', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2. Gaussian Naive Bayes: Learning the Electrical Distributions

Gaussian NB learns the mean and variance of each electrical feature for each defect type. Classification then asks: given these measurements, which defect type's distribution best explains what we observed?

In [ ]:
le = LabelEncoder()
X = df[feature_cols].values
y = le.fit_transform(df['defect_type'].values)
class_names = le.classes_

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

gnb = GaussianNB()
gnb.fit(X_train_s, y_train)
y_pred = gnb.predict(X_test_s)

print('Gaussian NB — Silicon Defect Classification:')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=class_names))

cv = cross_val_score(gnb, X_train_s, y_train, cv=StratifiedKFold(5))
print(f'5-Fold CV: {cv.mean():.3f} ± {cv.std():.3f}')

In [ ]:
# Visualize learned class means (the model's understanding of each defect type)
means_df = pd.DataFrame(gnb.theta_, columns=feature_cols, index=class_names)

fig, ax = plt.subplots(figsize=(12, 5))
normalized_means = (means_df - means_df.mean()) / means_df.std()
sns.heatmap(normalized_means, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Gaussian NB: Learned Class Means (z-scored) — The Model\'s Electrical Fingerprint per Defect')
ax.set_xlabel('Electrical Feature')
ax.set_ylabel('Defect Type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. LDA: Finding the Discriminant Directions

LDA finds the linear combinations of electrical features that best separate the defect types. For 6 defect classes, LDA can find up to 5 discriminant directions. The first two are usually sufficient for visualization and capture the most discriminative information.

In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_s, y_train)
y_pred_lda = lda.predict(X_test_s)

print('LDA — Silicon Defect Classification:')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_lda):.3f}')
print()
print(classification_report(y_test, y_pred_lda, target_names=class_names))

# LDA 2D projection
lda_2d = LinearDiscriminantAnalysis(n_components=2)
X_lda_train = lda_2d.fit_transform(X_train_s, y_train)
X_lda_test = lda_2d.transform(X_test_s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for i, (name, col) in enumerate(zip(class_names, colors)):
    mask_train = y_train == i
    mask_test = y_test == i
    ax1.scatter(X_lda_train[mask_train, 0], X_lda_train[mask_train, 1], 
               c=[col], label=name, alpha=0.4, s=20)
    ax2.scatter(X_lda_test[mask_test, 0], X_lda_test[mask_test, 1], 
               c=[col], label=name, alpha=0.7, s=30)

ax1.set_title('LDA Discriminant Space — Training Data')
ax1.set_xlabel('LD1'); ax1.set_ylabel('LD2'); ax1.legend(fontsize=7)
ax2.set_title('LDA Discriminant Space — Test Data')
ax2.set_xlabel('LD1'); ax2.set_ylabel('LD2'); ax2.legend(fontsize=7)

plt.suptitle('LDA: Silicon Defect Types in Discriminant Space', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Confusion Matrix and Error Analysis

Understanding which defect types get confused with each other is as important as the overall accuracy. Confusion between similar defect types (e.g., gate oxide and thin oxide) is expected and physically meaningful.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

cm_gnb = confusion_matrix(y_test, y_pred, normalize='true')
cm_lda = confusion_matrix(y_test, y_pred_lda, normalize='true')

sns.heatmap(cm_gnb, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax1)
ax1.set_title(f'Gaussian NB (Acc={accuracy_score(y_test, y_pred):.3f})')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
plt.setp(ax1.get_xticklabels(), rotation=30, ha='right', fontsize=8)

sns.heatmap(cm_lda, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=ax2)
ax2.set_title(f'LDA (Acc={accuracy_score(y_test, y_pred_lda):.3f})')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right', fontsize=8)

plt.suptitle('Silicon Defect Classification: Normalized Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Summary

This notebook demonstrated how generative classifiers can automatically identify silicon defect types from electrical measurements.

**Engineering impact**: Manual defect classification requires an experienced engineer to review each failing die's test data — a process that takes minutes per die. An automated classifier can process thousands of dies in seconds and provide a probability distribution over defect types, helping engineers prioritize which failures to investigate first.

**The learned class means (the heatmap in Section 2) are the most valuable output** — they encode the model's understanding of each defect type's electrical signature and can be used to generate new test vectors that specifically target each failure mode.